In [2]:
"""
LIMPIEZA Y ESTANDARIZACION DE DATOS - SINAICA
Estacion: Vallarta (VAL)
Periodo: 1 al 31 de marzo de 2024

Este script parte del archivo original 'SINAICA_Marzo_2024_og.xlsx' (que en
realidad contiene el año completo 2024 para las 13 estaciones de la red) y
lo reduce a la base de datos exacta que se necesita para el analisis
"""

import pandas as pd
import numpy as np
import os

# RUTAS DE ENTRADA / SALIDA
RUTA_ENTRADA = "SINAICA Marzo 2024 og.xlsx"
RUTA_SALIDA = "SINAICA_VALmarzo2024_limpio.xlsx"

ESTACION_OBJETIVO = "VAL"
FECHA_INICIO = "2024-03-01 00:00:00"
FECHA_FIN = "2024-03-31 23:00:00"


# 1. CARGA DE DATOS
df = pd.read_excel(RUTA_ENTRADA, sheet_name="Data", engine="openpyxl")
# Se limpian los nombres de columnas para evitar errores al filtrar por nombre.
df.columns = [c.strip() for c in df.columns]


# 2. FILTRAR LA ESTACION Y EL PERIODO
df["DATE"] = pd.to_datetime(df["DATE"])

df_val = df[df["STATION"] == ESTACION_OBJETIVO].copy()
df_marzo = df_val[(df_val["DATE"] >= FECHA_INICIO) & (df_val["DATE"] <= FECHA_FIN)].copy()

n_esperado = 31 * 24  # 31 dias x 24 horas = 744 registros esperados (igual que en el PDF)


# 3. ELIMINAR CONTAMINANTES QUE LA ESTACION VAL NO MIDE
# La estacion VAL solo esta equipada para medir O3, PM10 y PM2.5. Las columnas NO, NO2, NOX, SO2 y CO
# ya que son columnas comunes a TODAS las estaciones de la red, pero en VAL vienen vacias al 100%
# No es un dato faltante por falla, es que el sensor no existe fisicamente en la estacion
columnas_no_medidas = ["NO", "NO2", "NOX", "SO2", "CO"]
# Verificacion: confirmar que en efecto estan vacias al 100% para VAL-marzo
for col in columnas_no_medidas:
    disponibilidad = df_marzo[col].notna().mean() * 100
    if disponibilidad > 0:
        print(f"AVISO: {col} tiene {disponibilidad:.1f}% de datos en VAL; revisar antes de eliminar.")

df_marzo = df_marzo.drop(columns=columnas_no_medidas)


# 4. RESTRINGIR EL DATASET A LAS VARIABLES DEL ALCANCE DEL ANALISIS
# Contaminantes: O3, PM10, PM2.5
# Meteorologicas: ET, IT, RH, WS, WD, PP, UVI
columnas_identificadoras = ["STATION", "DATE", "HOUR", "DIA", "MES"]
columnas_alcance_ = ["O3", "PM10", "PM2.5", "ET", "IT", "RH", "WS", "WD", "PP", "UVI"]

df_marzo = df_marzo[columnas_identificadoras + columnas_alcance_]


# 5. ELIMINAR VARIABLES DEL ALCANCE NO TIENEN DATOS
# La ficha tecnica dice que VAL deberia registrar RH, WD, PP y UVI, pero al
# revisar los datos reales de marzo 2024 esas 4 columnas vienen vacias al
# 100% (0 de 744 horas).
columnas_vacias_detectadas = [c for c in columnas_alcance_ if df_marzo[c].notna().sum() == 0]
df_marzo = df_marzo.drop(columns=columnas_vacias_detectadas)


# 6. ESTANDARIZAR TIPOS DE DATO
# Al venir de Excel con columnas mixtas (por ejemplo PM2.5 se carga como
# "object" por tener texto o vacios en otras estaciones),
# se fuerza explicitamente el tipo numerico en las variables de
# medicion y el tipo entero en las columnas de tiempo.
columnas_numericas_finales = [c for c in df_marzo.columns if c not in columnas_identificadoras]
for col in columnas_numericas_finales:
    df_marzo[col] = pd.to_numeric(df_marzo[col], errors="coerce")

df_marzo["HOUR"] = df_marzo["HOUR"].astype(int)
df_marzo["DIA"] = df_marzo["DIA"].astype(int)
df_marzo["MES"] = df_marzo["MES"].astype(int)


# 7. VALIDAR CONSISTENCIA TEMPORAL (SIN HUECOS NI DUPLICADOS)
# Se construye la secuencia horaria completa que DEBERIA existir
# (744 horas, del 2024-03-01 00:00 al 2024-03-31 23:00)
rango_horario_completo = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="h")
horas_faltantes = set(rango_horario_completo) - set(df_marzo["DATE"])
duplicados = df_marzo["DATE"].duplicated().sum()

df_marzo = df_marzo.sort_values("DATE").reset_index(drop=True)


# 8. VALIDAR RANGOS FISICAMENTE POSIBLES (DETECCION DE OUTLIERS)
# Ninguna concentracion de contaminante puede ser negativa. Si existiera un
# valor negativo, seria un error de captura y se convierte a NaN
columnas_concentracion = [c for c in ["O3", "PM10", "PM2.5"] if c in df_marzo.columns]
valores_negativos_detectados = 0
for col in columnas_concentracion:
    negativos = df_marzo[col] < 0
    valores_negativos_detectados += negativos.sum()
    df_marzo.loc[negativos, col] = np.nan


# 9. GUARDAR LA BASE LIMPIA
df_marzo.to_excel(RUTA_SALIDA, index=False, sheet_name="VAL_marzo2024")



# 10. RESUMEN FINAL DE LA LIMPIEZA
print("=" * 70)
print("RESUMEN DE LA LIMPIEZA - ESTACION VAL, MARZO 2024")
print("=" * 70)
print(f"Registros esperados (31 dias x 24 h): {n_esperado}")
print(f"Registros obtenidos tras el filtro:   {len(df_marzo)}")
print(f"Horas faltantes como fila:            {len(horas_faltantes)}")
print(f"Filas con fecha duplicada:             {duplicados}")
print(f"Valores negativos corregidos a NaN:    {valores_negativos_detectados}")
print(f"Columnas eliminadas (sensor inexistente en VAL): {columnas_no_medidas}")
print(f"Columnas eliminadas (fuera del alcance del PDF): ['ATM', 'RS']")
print(f"Columnas eliminadas (0% de datos reales en marzo): {columnas_vacias_detectadas}")
print(f"Columnas finales en la base limpia: {list(df_marzo.columns)}")
print()
print("Disponibilidad final por variable:")
for col in columnas_numericas_finales:
    if col in df_marzo.columns:
        disponibilidad = df_marzo[col].notna().mean() * 100
        dias_con_dato = df_marzo.loc[df_marzo[col].notna(), "DIA"].nunique()
        print(f"  {col:<8} {disponibilidad:5.1f}% disponible | {dias_con_dato} dias con al menos 1 dato")
print()
print(f"Archivo generado: {RUTA_SALIDA}")

RESUMEN DE LA LIMPIEZA - ESTACION VAL, MARZO 2024
Registros esperados (31 dias x 24 h): 744
Registros obtenidos tras el filtro:   744
Horas faltantes como fila:            0
Filas con fecha duplicada:             0
Valores negativos corregidos a NaN:    0
Columnas eliminadas (sensor inexistente en VAL): ['NO', 'NO2', 'NOX', 'SO2', 'CO']
Columnas eliminadas (fuera del alcance del PDF): ['ATM', 'RS']
Columnas eliminadas (0% de datos reales en marzo): ['RH', 'WD', 'PP', 'UVI']
Columnas finales en la base limpia: ['STATION', 'DATE', 'HOUR', 'DIA', 'MES', 'O3', 'PM10', 'PM2.5', 'ET', 'IT', 'WS']

Disponibilidad final por variable:
  O3        42.6% disponible | 14 dias con al menos 1 dato
  PM10      42.5% disponible | 14 dias con al menos 1 dato
  PM2.5     36.3% disponible | 14 dias con al menos 1 dato
  ET        42.6% disponible | 14 dias con al menos 1 dato
  IT        42.6% disponible | 14 dias con al menos 1 dato
  WS        33.5% disponible | 14 dias con al menos 1 dato

Archivo gen